# Arcadia morning: one dream wallpaper (Phase 1.5)

Runs `daily/morning_job.py`: run 4 dreams 12 candidates, the best is picked and "remembered" by Qwen2.5-1.5B, Qwen writes a painting prompt, SDXL paints it (canonical Corva as a style-only reference), and the text is set over it as desktop and phone wallpapers.

Launched every morning by `daily/morning.py`. Needs **GPU T4**, **Internet on**, and the private datasets **danieltgregory/arcadia-run4-model** and **danieltgregory/arcadia-run5-model** attached; `daily/production_model.txt` says which model dreams. Output: `/kaggle/working/morning/<date>/`.

## 1. Clone the repo

In [ ]:
import os
if not os.path.exists("/kaggle/working/Arcadia"):
    !git clone --depth 1 -b main https://github.com/DanielGregory/Arcadia /kaggle/working/Arcadia
%cd /kaggle/working/Arcadia
!git log --oneline -1

## 2. Install only what Kaggle is missing (torch is never reinstalled)

In [ ]:
import importlib.util, subprocess, sys

# The wallpaper needs these; a failure here should stop the run.
missing = [pkg for pkg in ("diffusers", "accelerate", "sentencepiece") if importlib.util.find_spec(pkg) is None]
print("installing:", missing or "nothing, all present")
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

# The Sky (dream embeddings) is optional: if this install fails, the morning still runs.
if importlib.util.find_spec("sentence_transformers") is None:
    ok = subprocess.call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"]) == 0
    print("sentence-transformers:", "installed" if ok else "NOT installed (the Sky will skip today)")

## 3. Find the model dataset and check the GPU

Kaggle has mounted attached datasets at more than one path over the years, so search for the files instead of hard-coding one.

In [ ]:
import glob, torch
# Which model dreams: daily/production_model.txt ("run4" or "run5"), unless a test run overrides it.
# Rolling back is a one-word edit of that file; both models' datasets are attached to every run.
MODEL_OVERRIDE = None   # set by `daily/morning.py --model run5` for a test run
MODEL = MODEL_OVERRIDE or open("daily/production_model.txt").read().strip()
ckpt = glob.glob(f"/kaggle/input/**/{MODEL}_best.pt", recursive=True)[0]
folder = os.path.dirname(ckpt)
train = next(p for p in (f"{folder}/{MODEL}_train.txt", f"{folder}/train.txt") if os.path.exists(p))
print("model:", MODEL, ckpt, train, sep="\n")
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

## 4. Make the morning

`check=True`: a failure fails the whole Kaggle run, instead of finishing as "complete" with nothing made.

In [ ]:
subprocess.run([sys.executable, "-m", "daily.morning_job", "--ckpt", ckpt, "--train", train,
                "--out", "/kaggle/working/morning", "--run", MODEL], check=True)

## 5. Show it

In [ ]:
for path in glob.glob("/kaggle/working/morning/*/morning.md"):
    print(open(path).read())